# P1 signal audit

In [ ]:
# Check the if the dataset is available

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

RESOURCE_DIR_KIRMIZI = Path("dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Kirmizi_Pistachio")
RESOURCE_DIR_SIIRT    = Path("dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Siirt_Pistachio")

for name, folder in [("Kirmizi", RESOURCE_DIR_KIRMIZI), ("Siirt", RESOURCE_DIR_SIIRT)]:
    if folder.is_dir():
        print(f"OK   {name}: {folder.resolve()}")
    else:
        print(f"MISS {name}: {folder.resolve()}")



# Cap how many images to load per class
LIMIT = 5 # None = load everyting

records = []
for name, folder in [("Kirmizi", RESOURCE_DIR_KIRMIZI), ("Siirt", RESOURCE_DIR_SIIRT)]:
    count = 0
    for path in sorted(folder.rglob("*")):
        if path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}:
            continue
        if LIMIT is not None and count >= LIMIT:
            break
        with Image.open(path) as im:
            arr = np.asarray(im.convert("RGB"))
        records.append({"class": name, "path": path, "arr": arr})
        count += 1

print(f"Loaded {len(records)} images (LIMIT={LIMIT} per class)")

OK   Kirmizi: /home/vibe/skole/IDIG4120/p0/dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Kirmizi_Pistachio
OK   Siirt: /home/vibe/skole/IDIG4120/p0/dataset/Pistachio_Image_Dataset/Pistachio_Image_Dataset/Siirt_Pistachio
Loaded 10 images (LIMIT=5 per class)


# Format, array shape, data type, units, range and byte size

In [ ]:
# 1. Format (from magic bytes, not the extension)
def detect_format(path):
    with open(path, "rb") as fh:
        head = fh.read(8)
    if head.startswith(b"\xff\xd8\xff"):
        return "JPEG"
    if head.startswith(b"\x89PNG"):
        return "PNG"
    return "other"

formats = {r["class"]: {} for r in records}
for r in records:
    fmt = detect_format(r["path"])
    formats[r["class"]][fmt] = formats[r["class"]].get(fmt, 0) + 1
print("1. Format:", formats)

# 2. Array shape
shapes = {r["arr"].shape for r in records}
print("2. Per-image (H, W, C):", shapes)

# 3. Data type
print("3. Data type:", records[0]["arr"].dtype)

# 4. Units
print("4. Units: 8-bit intensity levels 0-255")

# 5. Range (per channel, across the loaded subset)
stacked = np.stack([r["arr"] for r in records])
for i, ch in enumerate("RGB"):
    print(f"5. {ch}: min={stacked[..., i].min()} max={stacked[..., i].max()}")

# 6. Byte size
disk_total = sum(r["path"].stat().st_size for r in records)
mem_total = sum(r["arr"].nbytes for r in records)
print(f"6. On disk: {disk_total} bytes | In memory: {mem_total} bytes")

1. Format: {'Kirmizi': {'JPEG': 5}, 'Siirt': {'JPEG': 5}}
2. Per-image (H, W, C): {(600, 600, 3)}
3. Data type: uint8
4. Units: 8-bit intensity levels 0-255; physical units NOT AVAILABLE
5. R: min=0 max=255
5. G: min=0 max=255
5. B: min=0 max=255
6. On disk: 242751 bytes | In memory: 10800000 bytes
